# Aliquot Sequences

For a positive integer $n$, let $s(n)$ be the **aliquot sum** of $n$: the sum of its proper divisors (all positive divisors except $n$ itself). The aliquot sequence starting at $n$ is

$$n,\ s(n),\ s(s(n)),\ \dots$$

Each sequence either reaches $0$ (after passing through a prime and then $1$), falls into a cycle (perfect numbers, amicable pairs, sociable cycles), or keeps going for as long as we are willing to compute.

## Changes in this version

**Bug fixes**

1. The original prime factorisation treated whatever was left after dividing by all primes below $10^6$ as a prime. Since `val_limit` is $10^{14}$, the remainder can be a product of two primes larger than $10^6$ (for example, $1000003 \times 1000033$ was treated as a prime), which made $\sigma(n)$ incorrect. We now use the Miller–Rabin test to check whether the remainder is prime, and Pollard's rho method to keep factorising it if it is composite.
2. The original `aliquot_basic` used `return print(...)`, which actually returns `None`; it now returns the list. It also did not check for cycles and never used `val_limit`: for example, starting at 562 it kept alternating between 284 and 220 until it reached 30 iterations. It now stops when a value repeats, when it reaches 0, or when a value exceeds the limit.

**Speed improvements**

3. The primes are now generated with the sieve of Eratosthenes, which reduced the time to find all primes below $10^6$ from about 3 seconds to about 0.1 seconds.
4. The factorisation now returns a dictionary `{prime: power}` directly, so the factors no longer need to be counted separately.
5. Values of $s(n)$ are now cached. Many sequences reach the same numbers, so running a large number of sequences is much faster.

**Code structure and verification**

6. `aliquot()` now returns `(sequence, status, cycle)`. Using this, the `classify` and `report` functions count how many sequences terminate, enter a cycle or remain undetermined for different values of $N$, the iteration limit and the value limit.
7. The fast method was checked against the slow method: the results agree for every $n < 2000$.
8. Imports now use the form `import math`, and numpy and matplotlib were removed because they were not used.

In [ ]:
# basic modules being imported
import math
import random
import time

## A first (inefficient) method

The simplest way to compute $s(n)$ is to test every integer below $n$ and add up the ones that divide it. We keep this version because it is easy to trust, so we can use it later to check the faster method.

In [ ]:
# -----   inefficient method for calculation   -----

def sum_proper_divisors_basic(n):
    '''
        Adds together every proper divisor of n by testing
        every integer from 1 to n - 1.
    '''
    total = 0
    for j in range(1, n):
        if n % j == 0:
            total += j
    return total


def aliquot_basic(x, val_limit = 10**9, iter_limit = 30):
    '''
        Function that takes a positive integer x and creates its
        aliquot sequence.
        val_limit stops the function if a term goes above it
        iter_limit stops the function after that many terms
        Function returns the list of terms of the aliquot sequence
    '''

    # the sequence starts with x itself
    aliquot_list = [x]

    while len(aliquot_list) < iter_limit:

        # replacing the old value with the new one
        x = sum_proper_divisors_basic(x)
        aliquot_list.append(x)

        # stops if it reaches 0, if a value repeats (a cycle),
        # or if the value gets too big
        if x == 0 or x in aliquot_list[:-1] or x > val_limit:
            break

    return aliquot_list

In [ ]:
print(aliquot_basic(562))
print(aliquot_basic(12))

# A more efficient way to calculate the aliquot sequence

Our first method is slow: checking every positive integer below the target number involves lots of unnecessary tests, which leads to long calculation times for large starting numbers and long sequences. We therefore looked into more efficient ways to calculate $s(n)$, in particular the part on multiplicative functions mentioned in the Project Booklet.

### Multiplicative functions

A function $f$ on the positive integers is **multiplicative** if $f(1) = 1$ and, whenever $a$ and $b$ are coprime ($\gcd(a, b) = 1$),

$$f(ab) = f(a)f(b).$$

If $a$ and $b$ are coprime, their prime factors are different. Each divisor of $ab$ can therefore be made in exactly one way by multiplying a divisor of $a$ by a divisor of $b$. Multiplying the two sums of divisors gives all these products once, so

$$\sigma(a)\sigma(b)=\sigma(ab).$$

Also $\sigma(1)=1$, so the function is multiplicative.

Using the multiplicative property, we can split any number into its prime powers and calculate the sum of divisors of each separately, for example:

$$
\begin{align*}
\sigma(12) &= \sigma(2^2)\,\sigma(3) \\
&= (1 + 2 + 4)(1 + 3) \\
&= (7)(4) = 28
\end{align*}
$$

To get the aliquot sum $s(n)$, we subtract the original number from the sum of divisors:

$$s(12) = 28 - 12 = 16$$

The sum of divisors of a prime power is a geometric series, $\sigma(p^a) = 1 + p + \dots + p^a = \frac{p^{a+1}-1}{p-1}$. So for a positive integer $n = p_1^{a_1} p_2^{a_2} \cdots p_k^{a_k}$:

$$\sigma(n) = \prod_{i=1}^k\frac{p_i^{a_i+1}-1}{p_i-1}, \qquad s(n) = \sigma(n) - n.$$

### Applying the multiplicative function

To put this method into practice, we first have to split any given number into its prime factors. We generate a list of primes up to a set limit, then use that list to factorise the number.

#### Generating the primes: the sieve of Eratosthenes

To generate the primes, we use a sieve: we start by assuming every number is prime, and each time we find a prime $p$ we cross out all of its multiples. We only need to start crossing out at $p^2$, because any smaller multiple of $p$ has a smaller prime factor and has already been crossed out.

In [ ]:
def primes_up_to(m):
    '''
        Returns a list of all primes up to and including m,
        using the sieve of Eratosthenes.
    '''

    # initially every number from 2 upwards is assumed to be prime
    is_prime = [True] * (m + 1)
    is_prime[0] = is_prime[1] = False

    # only need primes up to sqrt(m) to cross out every composite number
    for p in range(2, math.isqrt(m) + 1):
        if is_prime[p]:

            # crossing out the multiples of p, starting at p*p
            for j in range(p * p, m + 1, p):
                is_prime[j] = False

    return [p for p in range(2, m + 1) if is_prime[p]]


PRIME_LIMIT = 10**6

t0 = time.perf_counter()
primes = primes_up_to(PRIME_LIMIT)
print(len(primes), "primes below", PRIME_LIMIT, "found in", round(time.perf_counter() - t0, 2), "s")

#### Factorising larger numbers

A list of primes up to $10^6$ is not enough to factorise every number in our experiments. If the remaining number has two prime factors greater than $10^6$, neither will be found in the list.

We use the Miller–Rabin test to check the remaining number. If it is composite, Pollard's rho method finds a factor by calculating gcds of differences between terms of a modular sequence. The primality test uses twelve fixed bases, which are sufficient for the range of integers in these experiments.

In [ ]:
def is_prime(n):
    '''
        Deterministic Miller-Rabin test, correct for all n < 3.3e24.
    '''
    if n < 2:
        return False

    bases = (2, 3, 5, 7, 11, 13, 17, 19, 23, 29, 31, 37)

    # small cases: n is one of the bases or divisible by one
    for p in bases:
        if n % p == 0:
            return n == p

    # writing n - 1 = 2^t * d with d odd
    d, t = n - 1, 0
    while d % 2 == 0:
        d //= 2
        t += 1

    for a in bases:
        x = pow(a, d, n)
        if x == 1 or x == n - 1:
            continue

        # squaring up to t - 1 times, looking for -1
        found = False
        for i in range(t - 1):
            x = x * x % n
            if x == n - 1:
                found = True
                break
        if not found:
            # never reached -1, so a proves n is composite
            return False

    return True


def pollard_rho(n):
    '''
        Returns a non-trivial factor of the composite number n.
    '''
    if n % 2 == 0:
        return 2

    while True:
        c = random.randrange(1, n)
        x = y = random.randrange(2, n)
        d = 1

        # x moves one step, y moves two steps
        while d == 1:
            x = (x * x + c) % n
            y = (y * y + c) % n
            y = (y * y + c) % n
            d = math.gcd(abs(x - y), n)

        # d == n means this choice of c failed, so try another
        if d != n:
            return d

In [ ]:
def prime_factorisation(n):
    '''
        Returns a dictionary {prime: power} for the positive integer n.
    '''

    # empty dictionary for the factors to be added to
    factors = {}

    for p in primes:

        # checks if the current prime squared is greater than n
        if p * p > n:
            break

        # divides out p as many times as possible, counting the power
        while n % p == 0:
            if p in factors:
                factors[p] += 1
            else:
                factors[p] = 1
            n //= p

    # whatever is left has no small prime factors, but may still be composite
    leftovers = [n] if n > 1 else []

    while leftovers:
        m = leftovers.pop()
        if is_prime(m):
            if m in factors:
                factors[m] += 1
            else:
                factors[m] = 1
        else:
            # split m into two smaller parts and deal with each of them
            d = pollard_rho(m)
            leftovers += [d, m // d]

    return factors


print(prime_factorisation(12))
print(prime_factorisation(1000003 * 1000033))
print(prime_factorisation(2**10 * 999983**2 * 1000003))

Now we can compute $s(n)$ with the formula. Many different starting values run into the same numbers (for example every sequence that reaches a prime then goes to $1$ and $0$), so we store every value of $s(n)$ we have already computed in a dictionary and look it up instead of factorising again.

In [ ]:
# dictionary of values of s(n) we have already worked out
s_cache = {}

def s(n):
    '''
        Aliquot sum of n: the sum of the proper divisors of n.
    '''
    if n in s_cache:
        return s_cache[n]

    # using the formula for geometric series for each prime power
    sigma = 1
    for p, power in prime_factorisation(n).items():
        sigma *= (p**(power + 1) - 1) // (p - 1)

    # subtracting n to only count the proper divisors
    s_cache[n] = sigma - n
    return sigma - n

Before relying on the faster method we check it against the slow but obviously correct one.

In [ ]:
# comparing the two methods for every n below 2000
mismatches = [n for n in range(1, 2000) if s(n) != sum_proper_divisors_basic(n)]
print("values where the two methods disagree:", mismatches)

print("s(1) =", s(1), " s(6) =", s(6), " s(12) =", s(12), " s(220) =", s(220), " s(284) =", s(284))

## Calculating the sequence

We store the terms in a list and stop if a term is repeated, if it reaches zero, or if either of the limits is reached. The function also returns the reason for stopping so we can count the results.

In [ ]:
TERMINATES, CYCLE, ITER_LIMIT, VALUE_LIMIT = "terminates", "cycle", "iteration limit", "value limit"

def aliquot(x, val_limit = 10**14, iter_limit = 50):
    '''
        Computes the aliquot sequence starting at x.
        Returns (sequence, status, cycle), where cycle is the list of
        values in the cycle if status is CYCLE, and None otherwise.
    '''

    aliquot_list = [x]

    while True:
        x = aliquot_list[-1]

        # checking each stopping condition
        if x == 0:
            return aliquot_list, TERMINATES, None
        if x > val_limit:
            return aliquot_list, VALUE_LIMIT, None
        if len(aliquot_list) >= iter_limit:
            return aliquot_list, ITER_LIMIT, None

        x = s(x)

        # check to see if the new value is a repeat
        if x in aliquot_list:
            return aliquot_list, CYCLE, aliquot_list[aliquot_list.index(x):]

        aliquot_list.append(x)

In [ ]:
for n in [7, 12, 6, 220, 562, 12496, 276]:
    seq, status, cycle = aliquot(n)
    print(n, "->", status, "| cycle:", cycle, "| first terms:", seq[:10])

$6$ is perfect (a cycle of length 1), $220$ and $284$ are amicable (length 2), and $12496$ starts a sociable cycle of length 5. $276$ is the smallest starting value whose eventual behaviour is still unknown, so it hits our limits.

## Classifying all starting values below $N$

We now run the sequence for every starting value $1 \le n < N$ and count how often each outcome occurs. The cache of $s$ is cleared before each run so that the timings are fair.

In [ ]:
def classify(N, val_limit, iter_limit):
    '''
        Runs aliquot(n) for every 1 <= n < N.
        Returns a dictionary of the statuses and the time taken in seconds.
    '''
    s_cache.clear()
    t0 = time.perf_counter()
    counts = {TERMINATES: 0, CYCLE: 0, ITER_LIMIT: 0, VALUE_LIMIT: 0}
    for n in range(1, N):
        result = aliquot(n, val_limit, iter_limit)
        counts[result[1]] += 1
    return counts, time.perf_counter() - t0


def report(N, val_limit, iter_limit):
    counts, t = classify(N, val_limit, iter_limit)
    undecided = counts[ITER_LIMIT] + counts[VALUE_LIMIT]
    print("N =", N, " val_limit = 10^" + str(round(math.log10(val_limit))), " iter_limit =", iter_limit)
    print("   terminates:", counts[TERMINATES], " cycle:", counts[CYCLE],
          " iteration limit:", counts[ITER_LIMIT], " value limit:", counts[VALUE_LIMIT])
    print("   undecided:", round(100 * undecided / (N - 1), 2), "%   time:", round(t, 1), "s")

In [ ]:
# Varying the iteration limit
for iter_limit in [30, 100, 300, 1000]:
    report(20000, 10**9, iter_limit)

In [ ]:
# Varying the value limit
for e in [9, 12, 15, 18]:
    report(20000, 10**e, 1000)

In [ ]:
# Varying N
for N in [20000, 50000, 100000]:
    report(N, 10**15, 1000)